In [1]:
import numpy as np
import h5py

In [2]:
# load catalog
metacal = h5py.File('/pscratch/sd/a/athomsen/y3/cats/DESY3_metacal_v03-004.h5','r')

# load indexes
index = h5py.File('/pscratch/sd/a/athomsen/y3/cats/DESY3_indexcat.h5','r')

In [3]:
# load mask
mask = np.array(index['index']['select'])

# Load weights
w = np.array(metacal['catalog']['unsheared']['weight'])
w_1p = np.array(metacal['catalog']['sheared_1p']['weight'])
w_2p = np.array(metacal['catalog']['sheared_2p']['weight'])
w_1m = np.array(metacal['catalog']['sheared_1m']['weight'])
w_2m = np.array(metacal['catalog']['sheared_2m']['weight'])

# load sheared ellipticities
e1_1m = np.array(metacal['catalog']['sheared_1m']['e_1'])
e1_1p = np.array(metacal['catalog']['sheared_1p']['e_1'])
e2_2m = np.array(metacal['catalog']['sheared_2m']['e_2'])
e2_2p = np.array(metacal['catalog']['sheared_2p']['e_2'])

# compute responses
dg = 0.01
m1   = (np.average(e1_1p[mask],weights = w_1p[mask]) - np.average(e1_1m[mask],weights = w_1m[mask])) / (2.*dg)
m2   = (np.average(e2_2p[mask],weights = w_2p[mask]) - np.average(e2_2m[mask],weights = w_2m[mask])) / (2.*dg)

print ('rg',m1,m2)

# compute selection responses
# load sheared masks
mask_1m = np.array(index['index']['select_1m'])
mask_1p = np.array(index['index']['select_1p'])
mask_2m = np.array(index['index']['select_2m'])
mask_2p = np.array(index['index']['select_2p'])

# Load ellipticities
e1 = np.array(metacal['catalog']['unsheared']['e_1'])
e2 = np.array(metacal['catalog']['unsheared']['e_2'])


d_m1   = (np.average(e1[mask_1p],weights = w[mask_1p]) - np.average(e1[mask_1m],weights = w[mask_1m])) / (2.*dg)
d_m2   = (np.average(e2[mask_2p],weights = w[mask_2p]) - np.average(e2[mask_2m],weights = w[mask_2m])) / (2.*dg)
print ('rg+rs',m1+d_m1,m2+d_m2)




# you can also simply use the average responses in the catalogs, but they will be slightly less accurate,
# because they are defined as R11 =(e1_1p[mask] -e1_1m[mask]) / (2.*dg), whereas ideally you'd want to
# use the sheared weights as well. The difference is very small, though...
# Load responses
R1 = np.array(metacal['catalog']['unsheared']['R11'])
R2 = np.array(metacal['catalog']['unsheared']['R22'])
R1 = R1[mask]
R2 = R2[mask]

m1_   = np.average(R1,weights = w[mask])
m2_   = np.average(R2,weights = w[mask])
print ('rg',m1_,m2_)




# add responses together
m1   = (m1+d_m1)*np.ones(len(mask))
m2   = (m2+d_m2)*np.ones(len(mask))


# apply mask to shears
e1  = e1[mask]
e2  = e2[mask]

# compute mean shears
mean_e1 = np.average(e1,weights=w[mask])
mean_e2 = np.average(e2,weights=w[mask])


# compute sigma_e and n_eff
s   = (m1+m2)/2.
w = w[mask]

a1 = np.sum(w**2 * (e1-mean_e1)**2)
a2 = np.sum(w**2 * (e2-mean_e2)**2)
b  = np.sum(w**2)
c  = np.sum(w * s)
d  = np.sum(w)


sigma_e = (np.sqrt( (a1/c**2 + a2/c**2) * (d**2/b) / 2. ) )

area = 4143.
a    = np.sum(w)**2
b    = np.sum(w**2)
c    = area * 60. * 60.

neff = ( a/b/c )

print (sigma_e,neff)

rg 0.7096703611630152 0.7110857885006183
rg+rs 0.7175106173257537 0.719262760176361
rg 0.7086199762729599 0.7098185112945081
0.2614206120344434 5.592119612670559


In [4]:
metacal.close()
index.close()